<p class="lead">A molecular-dynamics run produces a file with thousands of structures in it, and a student's first instinct is to average something over them and compare it with the average of another run. That instinct produces most of the statistical errors in the simulation literature, because consecutive frames of a trajectory are not independent observations: a nanosecond of sampling can contain fewer than ten independent configurations. This unit takes six real simulations of the Trp-cage miniprotein — two temperatures, three replicas each — and treats their output exactly as what it is, a short, strongly autocorrelated multivariate time series. We measure how much information it actually contains, decide what to discard, compare the conditions in the two possible ways and watch them disagree by twenty-six orders of magnitude in the p-value, and then do the same careful accounting for fluctuations, projections, free-energy surfaces and cluster representatives.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad analiza la salida de una simulación de dinámica molecular como lo que es: una serie temporal corta y fuertemente autocorrelacionada. Con seis trayectorias reales de la miniproteína Trp-cage (dos temperaturas, tres réplicas cada una, 1 ns) se calculan la función de autocorrelación con <code>acf()</code>, el tiempo de autocorrelación integrado, la ineficiencia estadística y el tamaño de muestra efectivo; se promedia por bloques y se construye el gráfico de Flyvbjerg-Petersen; se detecta la equilibración con el promedio acumulado inverso y con el criterio de Chodera. El experimento central compara las dos condiciones de dos maneras: con las 3.000 tramas individuales (p del orden de 1e-26) y con las tres réplicas independientes (p = 0,69); la segunda es la honesta, porque la dispersión entre réplicas a la misma temperatura supera la diferencia entre temperaturas. Después se tratan las fluctuaciones por residuo como un problema de comparaciones múltiples con corrección BH, se reducen las coordenadas con <code>prcomp</code> y con un tICA implementado a mano, se lee un histograma bidimensional como superficie de energía libre con incertidumbre por bootstrap de bloques, y se agrupan las tramas para extraer estructuras representativas. La conclusión es siempre la misma: el número de réplicas, y no el número de tramas, determina lo que se puede afirmar.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Treat molecular-dynamics observables (RMSD, RMSF, radius of gyration, distances, hydrogen bonds, energies) as correlated time series: autocorrelation function, integrated autocorrelation time, statistical inefficiency, effective sample size and block averaging.</li>
<li>Assess equilibration and convergence (reverse cumulative averaging, Chodera's automated detection, replica agreement) and decide what to discard.</li>
<li>Compute uncertainties correctly across replicas and report them, applying the tests of Unit 8 to replica-level rather than frame-level data.</li>
<li>Reduce and cluster a conformational ensemble (PCA on Cartesian coordinates, tICA, k-means on the projection), read free-energy-like landscapes critically and attach sampling error to them.</li>
<li>Report energy decompositions of the MM/GBSA type with the statistics that hundreds of simultaneous per-residue comparisons demand.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running; every number quoted in the prose comes from a chunk of this same document. No simulation is run here and no trajectory file is parsed: the six runs arrive as frozen tables (<code>md_trpcage_observables.csv</code>, <code>md_trpcage_rmsf.csv</code>, <code>md_trpcage_ca_coords.npz</code>), so base R<sup class="c"><a href="#r30">[30]</a></sup> plus <code>data.table</code> and <code>ggplot2</code><sup class="c"><a href="#r26">[26]</a></sup> suffice — in a fresh Google Colab R runtime run <code>install.packages(c("data.table","patchwork"))</code> once. The data folder also contains <code>md_trpcage_strideA1.xtc</code> and <code>md_trpcage_protein.pdb</code>, a 100-frame protein-only trajectory for students who want to repeat the measurements with MDTraj<sup class="c"><a href="#r8">[8]</a></sup> or MDAnalysis<sup class="c"><a href="#r7">[7]</a></sup> in Python; R has no equivalent trajectory parser in this environment, which is why the coordinates are read from the <code>.npz</code> archive instead (section 1 explains how). The unit uses the estimation vocabulary of <a href="../U7_Estimation/index.html">Unit 7</a>, the testing vocabulary of <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> and the clustering of <a href="../U15_Clustering/index.html">Unit 15</a>.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(patchwork)
})
set.seed(0)
PAL  <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
RCOL <- c(A1 = "#2b5d9b", A2 = "#5b8bc9", A3 = "#8fb4e3",
          B1 = "#6a4c93", B2 = "#d2691e", B3 = "#a3421a")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ggplot2", as.character(packageVersion("ggplot2")),
    "| patchwork", as.character(packageVersion("patchwork")), "\n")



Everything below uses `stats::acf`, `t.test`, `p.adjust`, `prcomp` and `kmeans` from base R and the `stats` package: no simulation-analysis library is involved, because once the observables are tabulated the problem is ordinary time-series statistics.

## 1. What a production run leaves behind

<div class="box bridge"><h4>Domain context</h4>
<p><b>Molecular dynamics (MD)</b> integrates Newton's equations for every atom of a protein and its surrounding water,
with forces given by an empirical energy function called a <b>force field</b> (here AMBER
ff14SB<sup class="c"><a href="#r12">[12]</a></sup> with TIP3P-FB water). A <b>frame</b> is the set of atomic
coordinates saved at one instant — here every 2 picoseconds (ps), so a 1 nanosecond (ns) run leaves 500 frames. A
<b>replica</b> is an independent repetition of the same simulation started with different random velocities; replicas
differ because the dynamics are chaotic. The system here is <b>Trp-cage</b>, a designed 20-residue
miniprotein<sup class="c"><a href="#r13">[13]</a></sup> used as a test system since 2002 because it folds on the
microsecond scale<sup class="c"><a href="#r14">[14]</a></sup> — fast for a protein, still a thousand times longer than
the runs analysed here. The quantities extracted per frame are: <b>RMSD</b>, the root-mean-square deviation of the
C-alpha atoms from a reference structure after optimal superposition (a scalar measure of "how far the fold has
moved", in nm); <b>Rg</b>, the radius of gyration (compactness); the number of backbone <b>hydrogen bonds</b>;
<b>helix fraction</b>, the proportion of residues assigned to a helix by the DSSP
algorithm<sup class="c"><a href="#r15">[15]</a></sup>; a side-chain distance defining the hydrophobic "cage"; and the
thermostat and barostat readouts (instantaneous temperature, box volume, potential energy). <b>RMSF</b> is the
root-mean-square <i>fluctuation</i> of one residue around its own average position — a per-residue measure of
flexibility, not of displacement.</p></div>

The six runs were produced with OpenMM 8<sup class="c"><a href="#r10">[10]</a></sup><sup class="c"><a href="#r11">[11]</a></sup>
for this course: 4,810 atoms in a cubic box, 50 ps of NVT and 100 ps of NPT equilibration, then 1 ns of production
sampling in the NPT ensemble, at 300 K (condition A) and 340 K (condition B), three seeds each. The protocol follows the
current best-practice recommendations for foundations in molecular simulation<sup class="c"><a href="#r29">[29]</a></sup>.
The question they are meant to answer is the one a thesis student asks every week: *does the protein behave differently
at the higher temperature?* Everything below is about answering that honestly with 1 ns of sampling — and about
discovering how little 1 ns is.

<div class="box bridge"><h4>Substitution — reading a NumPy archive from R</h4>
<p>The superposed C-alpha coordinates are frozen in <code>md_trpcage_ca_coords.npz</code>, a NumPy archive. R has no
reader for it, and no MDTraj/MDAnalysis equivalent is installed, so the first chunk below writes one: an
<code>.npz</code> file is a ZIP container of <code>.npy</code> members, <code>unzip()</code> extracts them, and each
<code>.npy</code> is a short ASCII header (dtype, shape, byte order) followed by the raw buffer, which
<code>readBin()</code> reads directly. Twelve lines, no Python interpreter, and the arrays come out identical to
<code>numpy.load</code> — the same reader is used in <a href="../U15_Clustering/index.html">Unit 15</a>. The alternative
offered by the data folder, extracting coordinates from <code>md_trpcage_protein.pdb</code> plus the stride trajectory,
would give only one of the six runs, so the archive is the right source.</p></div>



In [ ]:
dp <- function(f) file.path("..", "data", f)      # knitr runs with cwd = this unit's folder
obs  <- fread(dp("md_trpcage_observables.csv"))
rmsf <- fread(dp("md_trpcage_rmsf.csv"))
RUNS <- c("A1", "A2", "A3", "B1", "B2", "B3"); DT <- 2      # ps between frames

read_npy <- function(path) {                       # minimal .npy reader
  con <- file(path, "rb"); on.exit(close(con)); readBin(con, "raw", 8)
  hlen  <- readBin(con, "integer", 1, size = 2, endian = "little")
  h     <- rawToChar(readBin(con, "raw", hlen))
  descr <- sub(".*'descr': *'([^']+)'.*", "\\1", h)
  dims  <- as.integer(Filter(nzchar, strsplit(
             gsub("[ )]", "", sub(".*'shape': *\\(([^)]*)\\).*", "\\1", h)), ",")[[1]]))
  x <- readBin(con, if (substring(descr, 2, 2) == "f") "double" else "integer",
               prod(dims), size = as.integer(substring(descr, 3)), endian = "little")
  if (length(dims) > 1) aperm(array(x, dim = rev(dims)), rev(seq_along(dims))) else x
}
nd <- file.path(tempdir(), "npz"); dir.create(nd, showWarnings = FALSE)
unzip(dp("md_trpcage_ca_coords.npz"), exdir = nd)
X <- do.call(rbind, lapply(RUNS, function(r) {     # 3000 frames x 60 coordinates (nm)
  a <- read_npy(file.path(nd, paste0(r, ".npy"))); t(apply(a, 1, function(m) as.vector(t(m)))) }))
time_ps <- read_npy(file.path(nd, "time_ps.npy"))
run_of  <- rep(RUNS, each = 500); time_of <- rep(time_ps, 6)
cat(sprintf("observables: %d x %d | rmsf: %d x %d | coordinates: %d frames x %d\n",
            nrow(obs), ncol(obs), nrow(rmsf), ncol(rmsf), nrow(X), ncol(X)))
cat("columns:", paste(names(obs), collapse = ", "), "\n")
knitr::kable(obs[, .(frames = .N, rmsd_mean = mean(rmsd_ca_nm), rmsd_sd = sd(rmsd_ca_nm)),
                 by = .(condition, replica)][order(condition, replica)], digits = 4)



Six runs of 500 frames, twenty residues, sixty coordinates per frame. The per-run means of the C-alpha RMSD already
contain the whole story of the unit: at 300 K the three replicas sit at 0.110, 0.160 and 0.251 nm — a spread of
0.142 nm between repetitions of the *same* simulation.



In [ ]:
obs[, run := paste0(condition, replica)]
p1 <- ggplot(obs, aes(time_ps, rmsd_ca_nm, colour = factor(replica))) +
  geom_line(linewidth = 0.3) +
  facet_wrap(~ condition, nrow = 1,
             labeller = labeller(condition = c(A = "condition A - 300 K", B = "condition B - 340 K"))) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange", "green")]), name = "replica") +
  labs(title = "The raw material: C-alpha RMSD against time",
       x = "time (ps)", y = "C-alpha RMSD (nm)") +
  theme(legend.position = "bottom")
print(p1)



<p class="caption"><b>Figure 1.</b> C-alpha RMSD against time for the six runs. Within a condition the three replicas
occupy clearly different bands — at 300 K the means are 0.110, 0.160 and 0.251 nm — and each trace wanders slowly, with
excursions lasting tens to hundreds of picoseconds. Neither panel shows unfolding (RMSD stays below 0.35 nm); what they
show is that each replica settled into its own region of conformational space and stayed there for the whole
nanosecond.</p>

Two features of Figure 1 decide the statistics of the whole unit. The traces are **slow**: a point is very similar to
the point 20 ps before it, so the 500 frames of a run are not 500 observations. And the replicas **disagree**: the
spread between replicas of the same condition (0.110 to 0.251 nm at 300 K) is larger than any difference the eye can
find between the two conditions. Sections 2 and 5 turn those two observations into numbers.

## 2. Frames are not samples: autocorrelation and effective sample size

For a stationary time series \(x_1,\dots,x_N\) with mean \(\bar{x}\), the **normalised autocorrelation function** at
lag \(k\) is

\[ C(k) = \frac{\frac{1}{N}\sum_{t=1}^{N-k}(x_t-\bar{x})(x_{t+k}-\bar{x})}{\frac{1}{N}\sum_{t=1}^{N}(x_t-\bar{x})^2}, \]

with \(C(0) = 1\). It answers "how much does a frame still tell me about a frame \(k\) steps later?". Summing it gives
the **statistical inefficiency**

\[ g = 1 + 2\sum_{k=1}^{k_{\max}} C(k), \]

the number of frames one must take to obtain one independent observation, and therefore the **effective sample size**
\(N_{\text{eff}} = N/g\). The correct standard error of the mean is \(\sigma/\sqrt{N_{\text{eff}}}\), not
\(\sigma/\sqrt{N}\) — an error of a factor \(\sqrt{g}\), which in this data is between 4 and 9. The **integrated
autocorrelation time** in physical units is \(\tau_{\text{int}} = \Delta t\,(\tfrac{1}{2} + \sum_{k\ge 1}C(k))\), so
that \(g = 2\tau_{\text{int}}/\Delta t\). The sum is truncated at the first zero crossing of \(C(k)\), because beyond it
the estimator is pure noise<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup>.

<div class="box bridge"><h4>Worked example: the autocorrelation of six numbers by hand</h4>
<p>Take the series 1, 2, 3, 4, 5, 6 (mean 3.5; deviations -2.5, -1.5, -0.5, +0.5, +1.5, +2.5). The denominator is
6.25 + 2.25 + 0.25 + 0.25 + 2.25 + 6.25 = 17.5, so the variance term is 17.5/6 = 2.9167. For lag 1 the five products
are (-2.5)(-1.5) = 3.75, (-1.5)(-0.5) = 0.75, (-0.5)(0.5) = -0.25, (0.5)(1.5) = 0.75 and (1.5)(2.5) = 3.75, summing to
8.75. Here the conventions part company: R's <code>acf()</code> divides that sum by \(N = 6\), giving
8.75/6 = 1.4583 and \(C(1) = 1.4583/2.9167 = 0.500\); the "unbiased" convention divides by \(N - k = 5\), giving
1.75 and \(C(1) = 0.600\). The Python edition of this unit uses the second and therefore reports autocorrelation
times about 10 % larger than the numbers below. Neither is wrong; the chunk prints both so that the difference is
visible rather than mysterious.</p></div>



In [ ]:
acf_vals <- function(x, nlags) as.numeric(acf(x, lag.max = nlags, plot = FALSE)$acf)
tau_g_neff <- function(x, dt = DT) {               # truncate the sum at the first zero crossing
  cc   <- acf_vals(x, length(x) %/% 2)
  zero <- if (any(cc <= 0)) which(cc <= 0)[1] - 1L else length(cc)
  s <- sum(cc[2:zero])
  c(tau = dt * (0.5 + s), g = 1 + 2 * s, n_eff = length(x) / (1 + 2 * s), zcross = (zero - 1) * dt)
}
toy <- c(1, 2, 3, 4, 5, 6); dev <- toy - mean(toy)
cat(sprintf("by hand, 1/N convention   : var %.4f, lag-1 covariance %.4f, C(1) = %.3f\n",
            sum(dev^2) / 6, sum(dev[-6] * dev[-1]) / 6,
            (sum(dev[-6] * dev[-1]) / 6) / (sum(dev^2) / 6)))
cat(sprintf("by hand, 1/(N-k) convention: lag-1 covariance %.4f, C(1) = %.3f\n",
            sum(dev[-6] * dev[-1]) / 5, (sum(dev[-6] * dev[-1]) / 5) / (sum(dev^2) / 6)))
cat(sprintf("by stats::acf()            : C(1) = %.3f\n", acf_vals(toy, 2)[2]))



The hand calculation reproduces `acf()` exactly in its own convention, 0.500, and shows where the other convention's
0.600 comes from. A monotone series is strongly autocorrelated, as it must be. The same estimator now goes to the real
trajectories.



In [ ]:
getcol <- function(r, col = "rmsd_ca_nm")
  obs[condition == substr(r, 1, 1) & replica == as.integer(substr(r, 2, 2))][[col]]
corr <- rbindlist(lapply(RUNS, function(r) {
  x <- getcol(r); t1 <- tau_g_neff(x); sd_ <- sd(x)
  data.table(run = r, rmsd_mean = mean(x), rmsd_sd = sd_, tau_rmsd_ps = t1[["tau"]],
             g = t1[["g"]], n_eff = t1[["n_eff"]], zcross_ps = t1[["zcross"]],
             tau_rg_ps = tau_g_neff(getcol(r, "rg_nm"))[["tau"]],
             se_naive = sd_ / sqrt(length(x)), se_correct = sd_ / sqrt(t1[["n_eff"]]),
             ratio = sqrt(t1[["g"]]))
}))
knitr::kable(corr, digits = 4)
knitr::kable(fread(dp("md_trpcage_replica_summary.csv"))[, .(condition, replica, tau_int_rmsd_ps, n_eff_rmsd)],
             digits = 2, caption = "The frozen summary table, computed with a different convention")



Two things deserve attention. First the physics: the integrated autocorrelation time of the C-alpha RMSD is 17.7 to
80.2 ps depending on the replica, so the 500 frames of a 1 ns run contain between 6.2 and 28.3 independent
observations. The naive standard error, 0.0005-0.0015 nm, under-estimates the real one by a factor \(\sqrt{g}\) of 4.2
to 9.0. (With \(\Delta t\) = 2 ps the columns \(\tau\) and \(g\) coincide numerically, because
\(g = 2\tau/\Delta t\).) Any error bar on an MD average that was computed as \(\sigma/\sqrt{N_{\text{frames}}}\) is
wrong by about that factor, and it is wrong in the dangerous direction.

Second the bookkeeping: the frozen summary file reports autocorrelation times about twice as large as ours
(137.96 ps against 68.38 ps for run A1) and effective sample sizes correspondingly smaller, because it uses the
convention \(\tau = \Delta t\,(1 + 2\sum_k C(k))\) — our \(g\) expressed in time units. Both appear in the literature
and in simulation software<sup class="c"><a href="#r31">[31]</a></sup>. The lesson is not that one is right — it is
that **"the autocorrelation time is 70 ps" is an incomplete statement**: give the definition, the truncation rule and
the frame interval, or give \(N_{\text{eff}}\) directly, which is unambiguous.



In [ ]:
lagdf <- rbindlist(lapply(RUNS, function(r)
  data.table(run = r, condition = substr(r, 1, 1),
             lag = (0:125) * DT, C = acf_vals(getcol(r), 125))))
pa <- ggplot(lagdf, aes(lag, C, colour = run, linetype = condition)) + geom_line(linewidth = 0.5) +
  geom_hline(yintercept = 0, linewidth = 0.3) +
  scale_colour_manual(values = RCOL, name = NULL) + scale_linetype(guide = "none") +
  labs(title = "Autocorrelation of the RMSD", x = "lag (ps)", y = "C(k)") +
  theme(legend.position = c(0.8, 0.72), legend.text = element_text(size = 6))
nedf <- rbind(data.table(run = corr$run, what = "independent samples", n = corr$n_eff),
              data.table(run = RUNS, what = "frames stored", n = 500))
pb <- ggplot(nedf, aes(run, n, fill = what)) + geom_col(position = "dodge", width = 0.7) +
  scale_y_log10() + scale_fill_manual(values = c(`frames stored` = "grey70",
                                                 `independent samples` = PAL[["blue"]]), name = NULL) +
  labs(title = "Frames stored vs independent samples", x = NULL, y = "samples (log)") +
  theme(legend.position = "bottom")
print(pa | pb)



<p class="caption"><b>Figure 2.</b> Left: the autocorrelation functions decay on very different time scales — runs A2
and B1 lose memory within about 80 ps, run B2 is still correlated at 176 ps. Right: the same information as a count.
Every run stored 500 frames; between 6.2 and 28.3 of them are statistically independent. The ratio is the number that
should govern every error bar in a simulation paper.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Compute the integrated autocorrelation time of <code>n_hbonds</code>, <code>helix_fraction</code> and
<code>potential_energy_kJ_mol</code> for run A1 with <code>tau_g_neff(getcol("A1", "n_hbonds"))</code> and so on. Which
observable decorrelates fastest, and why would you expect the potential energy of a 4,810-atom system to be the fastest
of all? What does that imply about using the potential energy as evidence that a simulation is "converged"?</p></div>

## 3. Block averaging and the Flyvbjerg-Petersen plot

The autocorrelation route needs a truncation rule. A route that needs none is **block averaging**: cut the series into
\(M\) consecutive blocks of length \(b\), average within each block, and compute the standard error of those \(M\) block
means. As \(b\) grows past the correlation time the block means become independent and the estimated standard error
stops growing; the plateau value is the honest error bar. Plotting it against \(b\) is the Flyvbjerg-Petersen
construction<sup class="c"><a href="#r3">[3]</a></sup>, and the shape of the curve is itself a diagnostic: if no
plateau is reached before the blocks become too few to estimate anything, the run is too short for the quantity being
measured.



In [ ]:
block_se <- function(x, b) {                       # standard error from blocks of b frames
  m <- length(x) %/% b
  if (m < 2) return(NA_real_)
  sd(colMeans(matrix(x[1:(m * b)], nrow = b))) / sqrt(m)
}
bs  <- c(1, 2, 5, 10, 20, 25, 50, 100)
blk <- as.data.frame(sapply(RUNS, function(r) sapply(bs, function(b) block_se(getcol(r), b))))
rownames(blk) <- paste0(bs * DT, " ps")
knitr::kable(blk, digits = 5, caption = "Standard error of the mean RMSD (nm) by block length")
cat("sigma/sqrt(N_eff) from section 2:",
    paste(sprintf("%s %.5f", corr$run, corr$se_correct), collapse = " | "), "\n")


In [ ]:
bdf <- melt(as.data.table(blk)[, b_ps := bs * DT], id.vars = "b_ps",
            variable.name = "run", value.name = "se")
p3 <- ggplot(bdf, aes(b_ps, se, colour = run)) + geom_line(linewidth = 0.6) + geom_point(size = 1.3) +
  geom_hline(data = corr, aes(yintercept = se_correct, colour = run), linetype = "dotted", linewidth = 0.4) +
  scale_x_log10() + scale_colour_manual(values = RCOL, name = NULL) +
  labs(title = "Flyvbjerg-Petersen plot: no plateau means the run is too short",
       x = "block length (ps)", y = "estimated standard error of the mean (nm)") +
  theme(legend.position = c(0.12, 0.72), legend.text = element_text(size = 6))
print(p3)



<p class="caption"><b>Figure 3.</b> Block-averaged standard error against block length for the six runs; dotted lines
mark the autocorrelation-based estimate sigma/sqrt(N_eff) of section 2. The curves rise steadily and none flattens
before 200 ps blocks, where only five blocks remain and the estimate itself becomes erratic — run A2 drops from
0.00173 to 0.00072 nm between the last two points, which is noise, not convergence. Where the two estimators can be
compared they agree well (0.01343 against 0.01336 nm for run B2, 0.00537 against 0.00572 nm for A1) and both stand a
factor of 4 to 9 above the naive sigma/sqrt(500) that most papers report.</p>

A curve that has not plateaued is not a failure of the method; it is the method reporting that one nanosecond does not
resolve the slowest motion contributing to this observable. The honest responses are to extend the simulation, to run
more replicas, or to report an error bar derived from the replicas rather than from within a single run — which is
what section 6 does, and what reviewers of simulation papers now
expect<sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r5">[5]</a></sup>.

## 4. Equilibration: deciding what to discard

The beginning of a production run is contaminated by relaxation from the starting structure. Two standard tools decide
where that contamination ends. The **reverse cumulative average** plots the mean of the series computed from time
\(t_0\) to the end, as a function of \(t_0\): while the system is still relaxing the curve drifts, and it flattens once
the remaining data are stationary. **Chodera's automated criterion**<sup class="c"><a href="#r4">[4]</a></sup>
formalises the trade-off — discarding data removes bias but costs samples — by choosing the \(t_0\) that maximises the
effective sample size \(N_{\text{eff}}(t_0) = (N - t_0)/g(t_0)\) of the retained portion.



In [ ]:
chodera_t0 <- function(x, dt = DT, step = 5) {
  n <- length(x); best <- c(0, -Inf)
  for (t0 in seq(0, as.integer(0.6 * n) - 1, by = step)) {
    seg <- x[(t0 + 1):n]
    if (length(seg) < 50) break
    ne <- length(seg) / tau_g_neff(seg)[["g"]]
    if (ne > best[2]) best <- c(t0, ne)
  }
  c(t0_ps = best[1] * dt, n_eff_after = best[2])
}
eq <- rbindlist(lapply(RUNS, function(r) {
  x <- getcol(r); b <- chodera_t0(x); kept <- x[(b[["t0_ps"]] / DT + 1):length(x)]
  data.table(run = r, t0_ps = b[["t0_ps"]], n_eff_after = b[["n_eff_after"]],
             mean_all = mean(x), mean_after = mean(kept), shift_nm = mean(kept) - mean(x))
}))
knitr::kable(eq, digits = 4)
cat(sprintf("mean |shift| caused by discarding: %.4f nm; largest %.4f nm (run %s)\n",
            mean(abs(eq$shift_nm)), max(abs(eq$shift_nm)), eq$run[which.max(abs(eq$shift_nm))]))


In [ ]:
t0s <- seq(0, 295, by = 5)
eqd <- rbindlist(lapply(RUNS, function(r) {
  x <- getcol(r)
  data.table(run = r, t0 = t0s * DT,
             mean_rest = sapply(t0s, function(t) mean(x[(t + 1):length(x)])),
             neff_rest = sapply(t0s, function(t) length(x) - t) /
                         sapply(t0s, function(t) tau_g_neff(x[(t + 1):length(x)])[["g"]]))
}))
pe1 <- ggplot(eqd, aes(t0, mean_rest, colour = run)) + geom_line(linewidth = 0.6) +
  scale_colour_manual(values = RCOL, name = NULL) +
  labs(title = "Reverse cumulative average", x = "discard everything before t0 (ps)",
       y = "mean RMSD of the rest (nm)") +
  theme(legend.position = c(0.85, 0.55), legend.text = element_text(size = 6))
pe2 <- ggplot(eqd, aes(t0, neff_rest, colour = run)) + geom_line(linewidth = 0.6) +
  scale_colour_manual(values = RCOL, guide = "none") +
  labs(title = "Chodera's criterion: maximise what is left", x = "t0 (ps)",
       y = "N_eff of the retained data")
print(pe1 | pe2)



<p class="caption"><b>Figure 4.</b> Left: reverse cumulative averages. Four of the six runs are flat from the first
frame — their 150 ps of equilibration was enough — while A1 drifts upwards and B3 downwards, by about 0.01 nm. Right:
the effective sample size of the retained data. Its maximum is at t0 = 0 for four runs, at 550 ps for A1 and at 300 ps
for B3, but the curves are noisy and shallow, which is the usual situation with so few independent samples: the
criterion is choosing between nearly equivalent options.</p>

The practical conclusion for this data set is that equilibration detection changes the means by at most 0.013 nm, an
order of magnitude less than the between-replica spread of 0.142 nm. We therefore keep all 500 production frames of
every run, and say so. That decision must be made and reported once, before the comparison between conditions —
choosing a discard point per condition after looking at the comparison is a way of manufacturing a difference, and is
the simulation version of the analytic flexibility that Unit 8 warns about.

## 5. The centrepiece: frame-level versus replica-level testing

Now the question the simulations were run to answer. Does the C-alpha RMSD differ between 300 K and 340 K? There are
two ways to ask it with the same data.

The **frame-level** test pools all 1,500 frames of condition A against all 1,500 frames of condition B and runs
Welch's t-test<sup class="c"><a href="#r25">[25]</a></sup>. The **replica-level** test first reduces each run to one
number — its mean — and compares three numbers against three. The first treats frames as independent samples, which
section 2 proved they are not: the effective number of independent configurations per condition is about 40 to 60, not
1,500, and even that counts correlated replicas separately. The second treats the replica as the experimental unit,
which it is: replicas differ by their random seed exactly as biological replicates differ by their subject.



In [ ]:
A <- obs[condition == "A", rmsd_ca_nm]; B <- obs[condition == "B", rmsd_ca_nm]
tf <- t.test(A, B)                                  # Welch by default in R
cat(sprintf("FRAME-LEVEL  : n = %d vs %d frames, means %.4f vs %.4f nm, Welch t = %.2f, p = %.3e\n",
            length(A), length(B), mean(A), mean(B), tf$statistic, tf$p.value))
mns <- dcast(obs[, .(m = mean(rmsd_ca_nm)), by = .(condition, replica)],
             condition ~ replica, value.var = "m")
a <- as.numeric(mns[condition == "A", -1]); b <- as.numeric(mns[condition == "B", -1])
tr <- t.test(a, b)
cat(sprintf("REPLICA-LEVEL: n = 3 vs 3 replica means %s vs %s\n",
            paste(round(a, 4), collapse = " "), paste(round(b, 4), collapse = " ")))
cat(sprintf("               means %.4f +/- %.4f vs %.4f +/- %.4f nm (SE over replicas), Welch t = %.3f, p = %.4f\n",
            mean(a), sd(a) / sqrt(3), mean(b), sd(b) / sqrt(3), tr$statistic, tr$p.value))
cat(sprintf("\nratio of p-values: %.3g\n", tr$p.value / tf$p.value))
cat(sprintf("within-condition spread at 300 K: %.3f nm; difference between condition means: %.3f nm\n",
            max(a) - min(a), abs(mean(a) - mean(b))))



The frame-level test returns p = 1.1e-26 and the replica-level test returns p = 0.69 on the same numbers — a ratio of
6e25. Only one of them answers the scientific question. The frame-level test is not "more sensitive"; it is answering a
different and uninteresting question — "do these 3,000 particular configurations come from the same distribution?" —
with an n inflated fifty-fold by correlation. Its p-value would keep shrinking if the frames were saved every 0.2 ps
instead of every 2 ps, without a single new configuration being sampled. That is the signature of a pseudo-replication
error<sup class="c"><a href="#r5">[5]</a></sup>.

The honest answer is the second one: with three replicas per condition, a difference of 0.020 nm cannot be
distinguished from the 0.142 nm spread between replicas at the same temperature. Trp-cage melts experimentally near
315 K but on the microsecond time scale<sup class="c"><a href="#r14">[14]</a></sup>, so one nanosecond at 340 K should
not show unfolding — and does not.



In [ ]:
ph <- ggplot(obs, aes(rmsd_ca_nm, fill = condition)) +
  geom_histogram(bins = 50, position = "identity", alpha = 0.6) +
  scale_fill_manual(values = c(A = PAL[["blue"]], B = PAL[["orange"]]),
                    labels = c("A (300 K)", "B (340 K)"), name = NULL) +
  labs(title = sprintf("Frame level: p = %.0e", tf$p.value), x = "C-alpha RMSD (nm)", y = "frames") +
  theme(legend.position = c(0.72, 0.8), legend.text = element_text(size = 6))
repdf <- obs[, .(m = mean(rmsd_ca_nm)), by = .(condition, replica)]
pr <- ggplot(repdf, aes(condition, m, colour = condition)) +
  stat_summary(fun = mean, fun.min = function(v) mean(v) - sd(v) / sqrt(3),
               fun.max = function(v) mean(v) + sd(v) / sqrt(3), colour = "black", linewidth = 0.5, width = 0.25,
               geom = "errorbar") +
  geom_point(size = 2.4) +
  scale_colour_manual(values = c(A = PAL[["blue"]], B = PAL[["orange"]]), guide = "none") +
  scale_x_discrete(labels = c("A (300 K)", "B (340 K)")) +
  labs(title = sprintf("Replica level: p = %.2f", tr$p.value), x = NULL, y = "mean RMSD per replica (nm)")
tempdf <- obs[, .(m = mean(temperature_K)), by = .(condition, replica)]
pt <- ggplot(tempdf, aes(condition, m, colour = condition)) + geom_point(size = 2.4) +
  scale_colour_manual(values = c(A = PAL[["blue"]], B = PAL[["orange"]]), guide = "none") +
  labs(title = "The control that does separate", x = NULL, y = "mean temperature (K)")
print(ph | pr | pt)



<p class="caption"><b>Figure 5.</b> The same comparison seen three ways. Left: pooled frames, two broad overlapping
histograms whose means differ by 0.020 nm — with 1,500 "samples" each, that difference is formally significant beyond
any threshold. Centre: the three replica means per condition, with the standard error over replicas; the error bars
overlap completely and the 300 K replicas straddle the 340 K ones. Right: the thermostat readout, where three replicas
per condition suffice for an unmistakable separation (300.42 +/- 0.22 K against 340.24 +/- 0.07 K).</p>

The right-hand panel is the control that makes the argument airtight. Replica-level testing is not insensitive: with
exactly three replicas per condition it detects the temperature difference at p = 5.8e-06 and the box-volume
difference (47.76 against 48.84 nm<sup>3</sup>) at p = 4.6e-07. It fails to detect a structural difference because
there is no structural difference large enough to see in 1 ns, not because n = 3 is hopeless — the thermostat and the
barostat<sup class="c"><a href="#r32">[32]</a></sup> impose their targets identically in every replica, which is
exactly the situation in which three replicas are plenty.



In [ ]:
cols <- c("rmsd_ca_nm", "rg_nm", "n_hbonds", "helix_fraction", "dist_W6_P12_nm",
          "temperature_K", "box_volume_nm3")
summ <- rbindlist(lapply(cols, function(cl) {
  fa <- obs[condition == "A"][[cl]]; fb <- obs[condition == "B"][[cl]]
  m  <- obs[, .(mm = mean(.SD[[1]])), by = .(condition, replica), .SDcols = cl]
  ra <- m[condition == "A", mm]; rb <- m[condition == "B", mm]
  data.table(observable = cl, A = mean(ra), A_se = sd(ra) / sqrt(3), B = mean(rb), B_se = sd(rb) / sqrt(3),
             p_frame = t.test(fa, fb)$p.value, p_replica = t.test(ra, rb)$p.value)
}))
knitr::kable(summ, digits = 4)



Reading the table row by row is the habit to acquire. Four of the five structural observables have frame-level
p-values between 2.4e-12 and 2.4e-30 and replica-level p-values between 0.36 and 0.88: every one of them would be
reported as a "significant effect of temperature" by the frame-level analysis, and none survives the honest test. The
fifth, the radius of gyration, is the instructive exception — it does not even reach significance at frame level
(p = 0.092), because the two conditions really do have the same compactness to within the noise of 1,500 frames. The
two instrument readouts behave in the opposite way: their frame-level p-values underflow to zero and their
replica-level p-values are 5.8e-06 and 4.6e-07, because the thermostat and the barostat impose them identically in
every replica. A results table of this shape, with both columns shown, is far more convincing to a reviewer than a
single column of stars<sup class="c"><a href="#r2">[2]</a></sup>.

<div class="exercise"><h4>Exercise 2</h4>
<p>Subsample each run by taking every fifth frame (<code>obs[seq(1, .N, by = 5)]</code> within each run, 100 frames per
run instead of 500) and repeat both tests. The replica-level p-value should barely move; the frame-level one should
rise by orders of magnitude. Explain in one sentence why the "significance" of the frame-level test is a property of
the output-writing frequency rather than of the physics, and what that implies about trajectories saved every
0.1 ps.</p></div>

## 6. Error bars from replicas

Once the replica is accepted as the unit of analysis, the recipe is short and is the one used in the group's papers.
Compute the observable per replica, average the replica values, and build the interval from the spread of those values
with \(n = \) number of replicas:

\[ \bar{x} = \frac{1}{n}\sum_i x_i, \qquad \mathrm{SE} = \frac{s}{\sqrt{n}}, \qquad
\text{CI}_{95\%} = \bar{x} \pm t_{0.975,\,n-1}\,\mathrm{SE}, \]

with \(t_{0.975,2} = 4.303\) for three replicas — more than twice the 1.96 of a large sample, which is what three
replicas cost. The frames inside a replica are used only to compute \(x_i\); they never enter the error bar.



In [ ]:
replica_ci <- function(v, conf = 0.95) {
  n <- length(v); se <- sd(v) / sqrt(n); h <- qt(0.5 + conf / 2, n - 1) * se
  c(mean = mean(v), se = se, lo = mean(v) - h, hi = mean(v) + h)
}
for (cd in c("A", "B")) {
  r <- replica_ci(if (cd == "A") a else b)
  cat(sprintf("condition %s: mean RMSD %.4f nm, SE %.4f, 95%% CI [%.4f, %.4f]  (t_0.975,2 = %.3f)\n",
              cd, r[["mean"]], r[["se"]], r[["lo"]], r[["hi"]], qt(0.975, 2)))
}
d <- mean(b) - mean(a); pooled <- sqrt(var(a) / 3 + var(b) / 3)
cat(sprintf("\ndifference B - A = %+.4f nm, SE %.4f, 95%% CI [%+.4f, %+.4f]\n",
            d, pooled, d - 4.303 * pooled, d + 4.303 * pooled))
sd_rep <- sd(c(a - mean(a), b - mean(b)))
for (target in c(0.02, 0.05, 0.10))
  cat(sprintf("to detect a %.2f nm difference with 80 %% power one needs about %.0f replicas per condition\n",
              target, ceiling(2 * (2.8 * sd_rep / target)^2)))



The 95 % interval for the 300 K mean runs from -0.005 to 0.352 nm: with three replicas and this much spread, the data
constrain the mean RMSD hardly at all (and the interval even reaches below zero, which an RMSD cannot be — a reminder
that a normal-theory interval on n = 3 is a crude instrument). The interval for the difference, [-0.218, +0.177] nm,
includes zero by a wide margin. The power calculation says what would be needed: with the observed between-replica
standard deviation of 0.050 nm, detecting a 0.05 nm difference at 80 % power requires about sixteen replicas of this
length, and a 0.02 nm difference about ninety-nine. That number — not the number of frames — is the one to put in a
computing-time proposal, and it is why the field has moved from "one long run" to "many independent
runs"<sup class="c"><a href="#r5">[5]</a></sup><sup class="c"><a href="#r21">[21]</a></sup>.

## 7. Per-residue fluctuations: twenty tests at once

The RMSF table gives one number per residue per run: 20 residues × 6 runs. Comparing the conditions residue by residue
is twenty simultaneous tests, and the multiple-comparison machinery of Unit 8 applies without modification
(`p.adjust(p, method = "BH")` is base R's Benjamini-Hochberg<sup class="c"><a href="#r24">[24]</a></sup>). With three
replicas per condition each individual test has almost no power, so the interesting output is not a list of
significant residues but the honest statement that the data do not support one.



In [ ]:
rf <- rmsf[, {
  ra <- rmsf_nm[condition == "A"]; rb <- rmsf_nm[condition == "B"]
  .(resname = resname[1], A = mean(ra), B = mean(rb), diff = mean(rb) - mean(ra),
    p = t.test(ra, rb)$p.value)
}, by = residue]
rf[, q_BH := p.adjust(p, method = "BH")]
cat(sprintf("mean C-alpha RMSF: A %.4f nm, B %.4f nm  (+%.0f %% at the higher temperature)\n",
            mean(rf$A), mean(rf$B), 100 * (mean(rf$B) / mean(rf$A) - 1)))
cat(sprintf("residues with raw p < 0.05: %d of 20 | with BH q < 0.05: %d | smallest raw p = %.3f, smallest q = %.3f\n",
            sum(rf$p < 0.05), sum(rf$q_BH < 0.05), min(rf$p), min(rf$q_BH)))
gm <- rmsf[, .(m = mean(rmsf_nm)), by = .(condition, replica)]
cat(sprintf("one global test on the mean RMSF per replica instead of twenty: p = %.3f\n",
            t.test(gm[condition == "A", m], gm[condition == "B", m])$p.value))
knitr::kable(head(rf[order(p)], 5), digits = 4)


In [ ]:
rbar <- rmsf[, .(m = mean(rmsf_nm), se = sd(rmsf_nm) / sqrt(3)), by = .(condition, residue)]
rbar <- merge(rbar, rf[, .(residue, resname)], by = "residue")
rbar[, lab := factor(sprintf("%s%d", resname, residue),
                     levels = sprintf("%s%d", rf$resname[order(rf$residue)], sort(rf$residue)))]
pf1 <- ggplot(rbar, aes(lab, m, fill = condition)) +
  geom_col(position = position_dodge(0.8), width = 0.75) +
  geom_errorbar(aes(ymin = m - se, ymax = m + se), position = position_dodge(0.8), width = 0.3, linewidth = 0.3) +
  scale_fill_manual(values = c(A = PAL[["blue"]], B = PAL[["orange"]]), name = NULL) +
  labs(title = "Per-residue fluctuation, mean +/- SE over 3 replicas", x = NULL, y = "C-alpha RMSF (nm)") +
  theme(axis.text.x = element_text(angle = 90, size = 5.5, vjust = 0.5), legend.position = "bottom")
pf2 <- ggplot(rf, aes(diff, -log10(p))) + geom_point(size = 1.6, colour = PAL[["blue"]]) +
  geom_hline(yintercept = -log10(0.05), linetype = "dotted") +
  annotate("text", x = min(rf$diff), y = -log10(0.05) + 0.06, label = "p = 0.05", hjust = 0, size = 2.5) +
  geom_text(data = head(rf[order(p)], 3), aes(label = sprintf("%s%d", resname, residue)),
            nudge_y = 0.05, size = 2.4) +
  labs(title = "Twenty tests, none surviving correction",
       x = "RMSF difference B - A (nm)", y = "-log10 p (raw)")
print(pf1 | pf2)



<p class="caption"><b>Figure 6.</b> Left: per-residue C-alpha RMSF with the standard error over the three replicas.
The familiar shape is there — the termini and the Gly-Gly-Pro loop fluctuate most — and the 340 K bars are on average
25 % higher (0.0743 against 0.0594 nm). Right: the same comparison as a volcano-style plot. Not one of the twenty
residues reaches raw p &lt; 0.05 (the smallest is p = 0.060 at Gly15, and the largest difference, +0.052 nm at the
C-terminal Ser20, has p = 0.129) and the smallest BH-adjusted q-value is 0.619.</p>

This is a textbook demonstration of two statistical facts at once. A systematic 25 % increase that is visible in every
replica is still not significant when each residue is tested with three values per group, because the between-replica
variance is large and a t-test on n = 3 needs a very large effect. And testing twenty residues multiplies the chance of
a false positive, so the Benjamini-Hochberg correction pushes the adjusted values further up. The right report is
therefore: "mean C-alpha RMSF increased from 0.059 to 0.074 nm at 340 K; with three replicas no individual residue
reached significance after FDR correction". Even the single global test on the replica mean RMSF — one test instead of
twenty, and therefore more power — returns p = 0.276, so the increase is not established by this data set either way.
Fewer tests, more power, and an answer to the question that was actually asked.

## 8. Projections: principal components and tICA

A frame of this system is 20 C-alpha positions, that is a point in 60-dimensional space; all six runs together are a
3,000 × 60 matrix, already superposed on the same reference. **Principal component analysis** finds the orthogonal
directions of largest positional variance — in the simulation literature this is called *essential
dynamics*<sup class="c"><a href="#r16">[16]</a></sup> — and projecting the frames onto the first two gives the picture
everybody draws. The caveat that must accompany that picture was published almost as early as the method itself: with
short trajectories the leading components are dominated by the slowest motion *that happened to occur*, and projections
of different runs can look like different states purely from lack of
sampling<sup class="c"><a href="#r17">[17]</a></sup><sup class="c"><a href="#r18">[18]</a></sup>.

**Time-lagged independent component analysis (tICA)** changes the objective: instead of maximising variance it
maximises autocorrelation at a chosen lag time, so its leading components are the *slowest* collective coordinates
rather than the widest<sup class="c"><a href="#r6">[6]</a></sup>. It is the standard first step of Markov-state
modelling<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>. No R package
implements it here, so the next chunk writes it: a generalised eigenvalue problem in fifteen lines of matrix algebra,
which is also the best way to understand what it does.



In [ ]:
Y   <- prcomp(X, center = TRUE, scale. = FALSE)$x        # 3000 x 60 projection
evr <- prcomp(X, center = TRUE, scale. = FALSE)$sdev^2
evr <- evr / sum(evr)
cat("PCA explained variance ratio:", round(evr[1:5], 3),
    sprintf("| first three together: %.3f\n", sum(evr[1:3])))
for (r in RUNS) {
  m <- run_of == r
  cat(sprintf("   %s  PC1 %+.3f (sd %.3f)   PC2 %+.3f\n", r, mean(Y[m, 1]), sd(Y[m, 1]), mean(Y[m, 2])))
}


In [ ]:
tica <- function(Z, groups, lag) {        # whiten with C(0), diagonalise the symmetrised C(lag)
  p <- ncol(Z); C0 <- matrix(0, p, p); Ct <- matrix(0, p, p); n <- 0
  for (g in unique(groups)) {
    W  <- scale(Z[groups == g, ], scale = FALSE)
    aa <- W[1:(nrow(W) - lag), ]; bb <- W[(lag + 1):nrow(W), ]
    C0 <- C0 + crossprod(aa) + crossprod(bb)
    Ct <- Ct + crossprod(aa, bb) + crossprod(bb, aa)
    n  <- n + 2 * nrow(aa)
  }
  C0 <- C0 / n; Ct <- Ct / n
  e  <- eigen(C0, symmetric = TRUE); w <- pmax(e$values, 1e-12)
  Wh <- e$vectors %*% diag(w^-0.5) %*% t(e$vectors)      # whitening matrix
  M  <- Wh %*% Ct %*% Wh
  ee <- eigen((M + t(M)) / 2, symmetric = TRUE); o <- order(-ee$values)
  list(ev = ee$values[o], proj = scale(Z, scale = FALSE) %*% (Wh %*% ee$vectors[, o]))
}
LAG <- 20                                                # frames = 40 ps
tc  <- tica(Y[, 1:10], run_of, LAG)
its <- -LAG * DT / log(pmin(pmax(tc$ev[1:4], 1e-9), 0.999))
cat(sprintf("tICA at lag %d ps: eigenvalues %s\n", LAG * DT, paste(round(tc$ev[1:4], 3), collapse = " ")))
cat("implied timescales (ps):", round(its, 1), " <- compare with the 1000 ps length of each run\n")


In [ ]:
pdf1 <- data.table(run = run_of, PC1 = Y[, 1], PC2 = Y[, 2], t1 = tc$proj[, 1], t2 = tc$proj[, 2])
pp1 <- ggplot(pdf1, aes(PC1, PC2, colour = run)) + geom_point(size = 0.35, alpha = 0.4) +
  scale_colour_manual(values = RCOL, name = NULL) +
  guides(colour = guide_legend(override.aes = list(size = 2))) +
  labs(title = "PCA: colour = run, not condition",
       x = sprintf("PC1 (%.0f %%)", 100 * evr[1]), y = sprintf("PC2 (%.0f %%)", 100 * evr[2])) +
  theme(legend.position = c(0.88, 0.72), legend.text = element_text(size = 6),
        legend.key.height = unit(7, "pt"))
pp2 <- ggplot(data.table(k = 1:10, cum = cumsum(evr[1:10])), aes(k, cum)) +
  geom_line(colour = PAL[["blue"]]) + geom_point(colour = PAL[["blue"]], size = 1.6) +
  labs(title = "Two components carry 68 %", x = "number of components", y = "cumulative variance explained")
pp3 <- ggplot(pdf1, aes(t1, t2, colour = run)) + geom_point(size = 0.35, alpha = 0.4) +
  scale_colour_manual(values = RCOL, guide = "none") +
  labs(title = "tICA at a 40 ps lag", x = "tIC1", y = "tIC2")
print(pp1 | pp2 | pp3)



<p class="caption"><b>Figure 7.</b> Left: the first two principal components explain 46 % and 21 % of the positional
variance, and the six runs occupy six nearly disjoint islands — run A3 sits at PC1 = +0.60 while A1 is at -0.33,
although both are at 300 K. Centre: the cumulative variance curve; three components reach 75 %. Right: the tICA
projection at a 40 ps lag, which separates the runs just as sharply because the slowest motion in this data set is
precisely the drift of each replica into its own basin.</p>

The left panel is the figure a student is most tempted to over-interpret. It does not show "the free-energy landscape
of Trp-cage at two temperatures"; it shows that each 1 ns run visited one region and that the regions differ between
runs of the *same* condition as much as between conditions. The tICA timescales say the same thing numerically: the
slowest implied timescale at a 40 ps lag is 177 ps, a fifth of the trajectory length, and estimates of implied
timescales become unreliable once they approach the simulation length. A converged ensemble would show the six clouds
overlapping, which is the practical convergence test to apply before any landscape is
published<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r21">[21]</a></sup>.

## 9. A two-dimensional histogram is not a free-energy surface until it has error bars

Given an equilibrium ensemble, the probability \(P(x,y)\) of being in a bin of a projection is related to a free energy
by \(F = -k_BT \ln P\) (up to an additive constant), with \(k_BT = 2.49\) kJ/mol at 300 K. Two conditions must hold for
that equation to mean anything: the sampling must be at equilibrium, and each bin must contain enough *independent*
frames for \(P\) to be estimated. Section 2 showed we have at most 28 independent frames per run, so the second
condition is already in trouble. The remedy is to estimate the uncertainty of \(F\) the same way the mean was
estimated — by resampling whole blocks of frames with the bootstrap<sup class="c"><a href="#r33">[33]</a></sup>, which
preserves the correlation inside a block.



In [ ]:
kT <- 0.0083145 * 300                                     # kJ/mol at 300 K
xe <- seq(min(Y[, 1]), max(Y[, 1]), length.out = 31)
ye <- seq(min(Y[, 2]), max(Y[, 2]), length.out = 31)
hist2 <- function(x, y) {
  ix <- pmin(pmax(findInterval(x, xe, rightmost.closed = TRUE), 1), 30)
  iy <- pmin(pmax(findInterval(y, ye, rightmost.closed = TRUE), 1), 30)
  matrix(as.numeric(table(factor(ix, 1:30), factor(iy, 1:30))), 30, 30)
}
H <- hist2(Y[, 1], Y[, 2]); P <- H / sum(H)
Fs <- -kT * log(ifelse(P > 0, P, NA)); Fs <- Fs - min(Fs, na.rm = TRUE)
blocks <- unlist(lapply(RUNS, function(r) split(which(run_of == r), rep(1:10, each = 50))),
                 recursive = FALSE)                       # 100 ps blocks
set.seed(0)
boot <- replicate(100, {
  pick <- unlist(blocks[sample(length(blocks), length(blocks), TRUE)])
  p <- hist2(Y[pick, 1], Y[pick, 2]); p <- p / sum(p)
  f <- -kT * log(ifelse(p > 0, p, NA)); f - min(f, na.rm = TRUE)
})
sdF  <- apply(boot, 1:2, function(v) sd(v, na.rm = TRUE))
well <- is.finite(Fs) & H >= 20
cat(sprintf("occupied bins: %d of 900 | bins with >= 20 frames: %d\n", sum(is.finite(Fs)), sum(well)))
cat(sprintf("apparent depth using ALL occupied bins: %.1f kJ/mol (this is the number not to quote)\n",
            max(Fs, na.rm = TRUE)))
cat(sprintf("free-energy range over well-populated bins: %.2f kJ/mol\n", max(Fs[well]) - min(Fs[well])))
cat(sprintf("bootstrap-over-blocks uncertainty there: median %.2f kJ/mol, maximum %.2f kJ/mol\n",
            median(sdF[well], na.rm = TRUE), max(sdF[well], na.rm = TRUE)))
cat(sprintf("ratio of the apparent range to its own uncertainty: %.1f\n",
            (max(Fs[well]) - min(Fs[well])) / median(sdF[well], na.rm = TRUE)))


In [ ]:
xc <- (xe[-31] + xe[-1]) / 2; yc <- (ye[-31] + ye[-1]) / 2
fdt <- data.table(x = rep(xc, 30), y = rep(yc, each = 30),
                  F = as.vector(Fs), sd = as.vector(ifelse(well, sdF, NA)))
pf_a <- ggplot(fdt[is.finite(F)], aes(x, y, fill = F)) + geom_raster() +
  scale_fill_viridis_c(direction = -1, name = "kJ/mol") +
  labs(title = "-kT ln P over the PCA projection", x = "PC1", y = "PC2")
pf_b <- ggplot(fdt[is.finite(sd)], aes(x, y, fill = sd)) + geom_raster() +
  scale_fill_viridis_c(option = "magma", name = "kJ/mol") +
  coord_cartesian(xlim = range(fdt$x), ylim = range(fdt$y)) +
  labs(title = "Bootstrap-over-blocks uncertainty", x = "PC1", y = "PC2")
print(pf_a | pf_b)



<p class="caption"><b>Figure 8.</b> Left: the two-dimensional histogram of the pooled 3,000 frames converted to
-kT ln P over its 366 occupied bins. It looks like a landscape with several minima separated by barriers of up to
9.9 kJ/mol. Right: the standard deviation of the same quantity over 100 bootstrap resamples of 100 ps blocks,
restricted to the 40 bins with at least 20 frames. The uncertainty there is about 1.1 kJ/mol over a usable range of
2.5 kJ/mol — the apparent structure is roughly two standard deviations deep — and the sparsely populated regions that
carry the deepest "barriers" are absent from the right panel altogether, because single blocks decide whether they are
occupied at all.</p>

Three rules follow, and they apply equally to the free-energy surfaces the group publishes for larger systems. Mask
bins below a stated occupancy instead of letting one frame create a minimum. Attach an uncertainty obtained by
resampling *blocks* (or, better, by comparing independent replicas) rather than frames. And never convert a histogram
to a free energy when the ensemble is not equilibrium sampling — after a biased or steered simulation the
transformation requires reweighting, which is the business of enhanced-sampling
methods<sup class="c"><a href="#r28">[28]</a></sup>.

## 10. Clustering frames and extracting representative structures

Clustering a trajectory serves a practical purpose: to replace 3,000 frames by a handful of structures that can be
drawn in a figure, docked, or submitted as supplementary coordinates. The algorithm is the k-means of Unit 15 applied
to the projection, and the representative of a cluster is the frame closest to its centre. The diagnostic that matters
is not the silhouette score but the cross-tabulation of clusters against runs.



In [ ]:
set.seed(0)
km <- kmeans(Y[, 1:5], centers = 4, nstart = 10)
ct <- table(run = run_of, cluster = km$cluster)
knitr::kable(as.data.frame.matrix(ct), caption = "Frames of each run per cluster")
for (k in 1:4) {
  m   <- which(km$cluster == k)
  dk  <- sqrt(rowSums((Y[m, 1:5] - matrix(km$centers[k, ], length(m), 5, byrow = TRUE))^2))
  rep_i <- m[which.min(dk)]
  cat(sprintf("cluster %d: %4d frames (%4.1f %%) | representative: run %s at %4d ps | RMSD there %.3f nm | runs with > 20 frames: %s\n",
              k, length(m), 100 * length(m) / nrow(Y), run_of[rep_i], time_of[rep_i],
              obs$rmsd_ca_nm[rep_i], paste(rownames(ct)[ct[, k] > 20], collapse = ", ")))
}


In [ ]:
ctd <- as.data.table(ct)
pcl1 <- ggplot(ctd, aes(factor(cluster), run, fill = N)) + geom_tile() +
  geom_text(aes(label = N), size = 2.6, colour = "white") +
  scale_fill_gradient(low = PAL[["purple"]], high = PAL[["blue"]], name = "frames") +
  labs(title = "Frames of each run per cluster", x = "cluster", y = NULL)
pcl2 <- ggplot(data.table(PC1 = Y[, 1], PC2 = Y[, 2], cl = factor(km$cluster)),
               aes(PC1, PC2, colour = cl)) + geom_point(size = 0.35, alpha = 0.35) +
  annotate("point", x = km$centers[, 1], y = km$centers[, 2], shape = 4, size = 2.4, colour = "black") +
  scale_colour_manual(values = unname(PAL), name = "cluster") +
  guides(colour = guide_legend(override.aes = list(size = 2))) +
  labs(title = "k-means (k = 4) on the first five PCs")
print(pcl1 | pcl2)



<p class="caption"><b>Figure 9.</b> Left: how the frames of each run distribute over the four clusters. Run A3 is
entirely in cluster 1, A2 almost entirely in cluster 4 (499/500), A1 almost entirely in cluster 3 (497/500) and B1
94 % in that same cluster; only B2 (187/313) and B3 (329/163) are genuinely split between two clusters. Right: the
same partition in the PC1-PC2 plane. The clusters are compact and well separated — and they are the replicas.</p>

A clustering that recovers the replica labels is reporting that the ensemble is not converged, not that the protein has
four metastable states. Published as "the four dominant conformational states of Trp-cage", it would be an artefact;
published as "each 1 ns replica remains within one basin, and the basins differ between replicas", it is a correct and
useful statement about sampling. The representative structures are still perfectly usable as illustrations — cluster 3,
sampled by A1, B1 and part of B3, is the most native-like with RMSD 0.106 nm at its representative frame — provided the
caption says they are representatives of *what was sampled* rather than of equilibrium populations. A cluster
population is only an equilibrium probability if the ensemble is converged, which here it is not.

## 11. Reporting an energy decomposition

Binding-free-energy estimates of the MM/PBSA and MM/GBSA family<sup class="c"><a href="#r9">[9]</a></sup> are computed
frame by frame and are usually accompanied by a per-residue decomposition. Everything in this unit applies to them,
and more sharply, because the numbers are large and their differences are small. Three requirements are not optional.
First, the error bar on a mean interaction energy must come from independent blocks or replicas, never from
\(\sigma/\sqrt{N_{\text{frames}}}\): energies decorrelate fast but the conformations that generate them do not.
Second, a per-residue table of 200 residues compared between two systems is 200 simultaneous tests and needs the
false-discovery-rate control of section 7; the usual presentation, a bar chart of the ten largest contributions, is a
selection effect by construction. Third, the absolute value of an MM/GBSA energy is not a binding free energy —
comparisons within one system and one protocol are defensible, comparisons across protocols are not. A reviewer who
sees "ligand A: -45.2 +/- 0.3 kcal/mol, ligand B: -43.8 +/- 0.3 kcal/mol, therefore A binds better" is reading a
standard error computed over correlated frames of a single trajectory, and the correct response is to ask for the
replicas.



In [ ]:
pe <- getcol("A1", "potential_energy_kJ_mol")
t_pe <- tau_g_neff(pe)
cat(sprintf("run A1 potential energy: mean %.0f kJ/mol, sd %.0f\n", mean(pe), sd(pe)))
cat(sprintf("  tau_int = %.1f ps, N_eff = %.1f of 500 frames\n", t_pe[["tau"]], t_pe[["n_eff"]]))
cat(sprintf("  naive SE = %.1f kJ/mol | autocorrelation-corrected SE = %.1f | 100 ps block SE = %.1f\n",
            sd(pe) / sqrt(500), sd(pe) / sqrt(t_pe[["n_eff"]]), block_se(pe, 50)))
cat(sprintf("  replica-level SE over the three 300 K runs = %.1f kJ/mol\n",
            sd(sapply(c("A1", "A2", "A3"), function(r) mean(getcol(r, "potential_energy_kJ_mol")))) / sqrt(3)))



The potential energy of the whole system is the instructive counterexample, and it is worth reading carefully. It
decorrelates in 1.8 ps, so 271 of the 500 frames of run A1 are effectively independent and the three within-run error
bars agree within a factor of 1.4 (12.2 kJ/mol naive, 16.5 autocorrelation-corrected, 12.9 from 100 ps blocks). The
replica-level standard error is *smaller*, 1.5 kJ/mol, because the total potential energy of 4,810 atoms is dominated
by the water box, which the three runs equilibrated to the same value (-79,538, -79,534 and -79,539 kJ/mol). So "use
replicas" is not a ritual that always inflates error bars: it is an instruction to match the error bar to the slowest
motion the quantity depends on. A total energy depends on fast solvent degrees of freedom and converges quickly; an
RMSD, an RMSF or an MM/GBSA *interaction* energy depends on the conformation of a few residues, whose relaxation is the
slow motion this unit has been measuring all along — and those are the quantities whose published error bars are
routinely a factor of five to ten too small.

## 12. Summary

Six real nanosecond trajectories of a 20-residue protein were analysed here with no simulation software, only the
frozen tables they produced and base R. The accounting came first: the integrated autocorrelation time of the C-alpha
RMSD is 17.7 to 80.2 ps depending on the replica, so each run of 500 stored frames contains between 6.2 and 28.3
independent configurations and the naive standard error is too small by a factor of 4.2 to 9.0. Block averaging agreed
with the autocorrelation estimate where both could be computed (0.01343 against 0.01336 nm for B2), and the
Flyvbjerg-Petersen curves never reached a plateau — the data telling us that 1 ns does not resolve the slowest
contributing motion. Equilibration detection by reverse cumulative averaging and by Chodera's effective-sample-size
criterion moved the means by at most 0.013 nm, so all frames were kept, and that decision was taken before any
comparison.

Then the central experiment. Pooling frames, the two temperatures differ in C-alpha RMSD with p = 1.1e-26; comparing
the three replica means per condition, the same data give p = 0.69, with condition means of 0.174 +/- 0.041 and
0.153 +/- 0.019 nm. The 300 K replicas alone span 0.110 to 0.251 nm, a spread seven times the difference between the
conditions. Three more structural observables behave identically — frame-level p-values from 2.4e-12 to 2.4e-30,
replica-level p-values from 0.36 to 0.60 — while the thermostat and barostat readouts are significant by both routes
(300.42 +/- 0.22 K against 340.24 +/- 0.07 K, p = 5.8e-06; 47.76 against 48.84 nm3, p = 4.6e-07), proving that n = 3 is
not the obstacle. A power calculation on the between-replica standard deviation of 0.050 nm says that detecting a
0.05 nm difference would need about sixteen replicas, and a 0.02 nm difference about ninety-nine.

The same discipline reshaped every downstream analysis. Per-residue RMSF rose by 25 % on average at 340 K (0.059 to
0.074 nm) yet no residue of twenty reached raw p &lt; 0.05, let alone an FDR-corrected one, and the single global test
on the replica means returns p = 0.276. PCA put the six runs in six nearly disjoint islands (PC1 means from -0.33 for
A1 to +0.60 for A3, both at 300 K) and tICA at a 40 ps lag separated them further, with a slowest implied timescale of
177 ps — a fifth of the run length. The -kT ln P surface built from the pooled frames showed apparent minima of
9.9 kJ/mol, but a bootstrap over 100 ps blocks gave an uncertainty near 1.1 kJ/mol over a defensible range of only
2.5 kJ/mol across the 40 bins holding at least 20 frames. k-means on the projection returned four clean clusters that
are, almost exactly, the six replicas. The potential energy was the counterexample that sharpens the rule: it
decorrelates in 1.8 ps and its replica-level error bar (1.5 kJ/mol) is *smaller* than its within-run one, because a
total energy is dominated by fast solvent motion rather than by the slow conformational relaxation that governs every
structural observable.

None of this means the simulations were useless: it means that what 1 ns × 6 supports is a description of what was
sampled, error bars built from replicas, and an honest statement of what could not be resolved. The three habits to
carry away are to quote \(N_{\text{eff}}\) whenever an MD average is reported, to make the replica the unit of analysis
in every test, and to put the uncertainty of a landscape or an energy decomposition next to the landscape
itself<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r27">[27]</a></sup>.
Microsecond studies of this same protein<sup class="c"><a href="#r19">[19]</a></sup><sup class="c"><a href="#r20">[20]</a></sup>
exist precisely because the honest analysis of a nanosecond says so little.

<div class="box bridge"><h4>Bridge</h4>
<p>Within this course: the confidence intervals and the bootstrap come from <a href="../U7_Estimation/index.html">Unit 7</a>,
the tests, power and FDR control from <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>, PCA from
<a href="../U13_Features_Dimensionality/index.html">Unit 13</a>, and k-means and the same <code>.npz</code> reader from
<a href="../U15_Clustering/index.html">Unit 15</a>. The previous unit,
<a href="../U19_Virtual_Screening/index.html">Unit 19</a>, treated the output of a screening pipeline with the same
attitude: correlated samples, honest error bars, and a benchmark interrogated rather than believed. The Python edition
of this unit, which reads the archive with <code>numpy.load</code> and uses scikit-learn for the projection, is
<a href="../../Python/U20_MD_Analytics/index.html">here</a>. In the sister courses: how these trajectories are
produced, from structure preparation to thermostats, is
<a href="../../../BGM324/05_Molecular_Dynamics/index.html">BGM324 unit 5 (molecular dynamics)</a>,
with the structural background in
<a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">BGM324 unit 2</a>;
the free-energy and docking context is in
<a href="../../../BGM324/04_Computer_Aided_Drug_Design/index.html">BGM324 unit 4</a>,
and the machine-learning background in
<a href="../../../BGM411/R/index.html">BGM411 (R edition)</a>.</p></div>

## 13. Self-assessment

<details><summary>1. A paper reports "RMSD = 0.21 +/- 0.01 nm (mean +/- SEM over 5,000 frames)". What is wrong, and what would you ask for?</summary>
<p>The standard error was computed as sigma/sqrt(N_frames), which assumes independent frames. With an autocorrelation
time of tens of picoseconds the effective sample size is one or two orders of magnitude smaller, so the quoted
uncertainty is too small by sqrt(g). Ask for the integrated autocorrelation time or N_eff, a block-averaging plot, or —
best — the per-replica means and an error bar built from them.</p></details>

<details><summary>2. Why does saving frames more often make a frame-level t-test "more significant" without adding information?</summary>
<p>Because the test's n is the number of saved frames, while the information content is set by the number of
independent configurations, N_eff = N/g. Halving the output interval doubles N and leaves N_eff unchanged, so the
p-value falls while nothing has been learnt. This is pseudo-replication.</p></details>

<details><summary>3. State the difference between the statistical inefficiency g and the integrated autocorrelation time, and why two analyses of the same run can report autocorrelation times differing by a factor of two.</summary>
<p>g = 1 + 2 sum_k C(k) is dimensionless and equals the number of frames per independent sample; tau_int = dt(1/2 +
sum_k C(k)) has units of time and satisfies g = 2 tau_int/dt. Some software reports dt(1 + 2 sum C(k)) = g dt instead,
which is exactly twice tau_int — the factor of two seen between our table and the frozen summary file. A second, smaller
source of disagreement is the normalisation of the autocorrelation estimator itself: R's <code>acf()</code> divides by
N, the "unbiased" convention by N - k. Always give the definition or report N_eff.</p></details>

<details><summary>4. Your Flyvbjerg-Petersen curve rises monotonically and never plateaus. What are your options?</summary>
<p>The run is shorter than the slowest motion affecting the observable. Extend it, add independent replicas and build
the error bar from the replica means, or restrict the claim to an observable that does decorrelate within the run.
Reporting the last computable block value as "the" error bar understates the uncertainty.</p></details>

<details><summary>5. Four of six runs need no equilibration discard and two need 300-550 ps. Should you discard the same amount from every run?</summary>
<p>Applying a common, pre-registered discard (for instance the first 100 ps of every run) is defensible and simple;
applying a per-run automated criterion is also defensible. What is not defensible is choosing per-run discards after
seeing how they affect the comparison between conditions. Decide the rule first, report it, and show that the
conclusion does not depend on it.</p></details>

<details><summary>6. With three replicas per condition the temperature difference is detected at p = 6e-06 but the RMSD difference is not detected at all. What does that prove?</summary>
<p>That n = 3 is not inherently powerless: when the effect is large relative to the between-replica spread, three
replicas suffice. The failure to detect an RMSD difference is therefore evidence that the structural effect is small
compared with replica-to-replica variation at 1 ns, not evidence that the test is too weak to be used.</p></details>

<details><summary>7. How many replicas would you request to detect a 0.05 nm difference in mean RMSD, and what did that estimate assume?</summary>
<p>About sixteen per condition, from the observed between-replica standard deviation of 0.050 nm, 80 % power and a
two-sided 5 % level. It assumes that the replica means are approximately normal and that the spread observed with
three replicas is a reasonable estimate — itself very uncertain with two degrees of freedom, so treat the number as an
order of magnitude.</p></details>

<details><summary>8. A PCA plot shows two separated clouds for two trajectories of the same system. Name two innocent explanations before "two conformational states".</summary>
<p>Insufficient sampling — each run relaxed into a different basin and never crossed back, which is exactly what
Figure 7 shows — and the projection itself, since the leading components are chosen to maximise the variance that
happens to be present, so any slow drift becomes PC1. A convergence check is whether independent replicas overlap in
the projection.</p></details>

<details><summary>9. What must accompany a -kT ln P landscape for it to be interpretable?</summary>
<p>The occupancy threshold used to mask sparse bins, an uncertainty estimated by resampling blocks or by comparing
replicas, a statement that the sampling is unbiased equilibrium sampling (or the reweighting scheme if it is not), and
the temperature used in kT. Here the uncertainty was about 1.1 kJ/mol against a usable depth of 2.5 kJ/mol, while the
unmasked figure suggested 9.9 kJ/mol.</p></details>

<details><summary>10. k-means on your trajectory projection returns clusters that coincide with the replicas. What do you report?</summary>
<p>That the ensemble is not converged: each replica remained in its own basin. The representative structures may still
be shown as illustrations of what was sampled, but cluster populations must not be presented as equilibrium
probabilities, and the number of clusters must not be presented as a number of metastable states.</p></details>

<details><summary>11. A collaborator sends a per-residue MM/GBSA decomposition for two ligands and asks which residues differ. What do you do first?</summary>
<p>Count the tests. A 200-residue table is 200 comparisons; control the false-discovery rate with
<code>p.adjust(..., method = "BH")</code>, use replica-level or block-level error bars rather than per-frame ones, and
refuse to interpret the ten largest bars as a finding, since selecting them after looking is a selection effect. Report
effect sizes with intervals, not stars.</p></details>

<details><summary>12. Section 11 found a replica-level error bar on the potential energy that was smaller than the within-run one. Does that contradict the rest of the unit?</summary>
<p>No. The replica-level error bar is the right one in both cases; it is simply small here, because the total potential
energy is dominated by solvent degrees of freedom that decorrelate in picoseconds and reach the same mean in every run.
The rule is to match the error bar to the slowest motion the quantity depends on, and for conformational observables
that motion is slower than one nanosecond.</p></details>

## Glossary

<dl class="gloss">
<dt>Frame</dt><dd>One set of coordinates saved from a trajectory; here one every 2 ps, 500 per run.</dd>
<dt>Replica</dt><dd>An independent repetition of a simulation with a different random seed; the experimental unit for statistics.</dd>
<dt>RMSD</dt><dd>Root-mean-square deviation of atomic positions from a reference after optimal superposition; a measure of how far a structure has moved.</dd>
<dt>RMSF</dt><dd>Root-mean-square fluctuation of a residue around its own mean position; a measure of local flexibility.</dd>
<dt>Autocorrelation function C(k)</dt><dd>Normalised correlation between a series and itself k steps later; C(0) = 1. Computed here with <code>stats::acf</code>.</dd>
<dt>Integrated autocorrelation time</dt><dd>tau_int = dt(1/2 + sum_k C(k)); the time scale over which the series loses memory.</dd>
<dt>Statistical inefficiency g</dt><dd>1 + 2 sum_k C(k) = 2 tau_int/dt; the number of frames needed per independent sample.</dd>
<dt>Effective sample size</dt><dd>N_eff = N/g; the number of independent observations a correlated series is worth.</dd>
<dt>Block averaging</dt><dd>Estimating the standard error from the means of consecutive blocks; the estimate plateaus once blocks exceed the correlation time.</dd>
<dt>Flyvbjerg-Petersen plot</dt><dd>Block-averaged standard error against block length; absence of a plateau means the run is too short.</dd>
<dt>Equilibration detection</dt><dd>Choosing the portion of a run to discard, e.g. by reverse cumulative averaging or by maximising the retained effective sample size.</dd>
<dt>Pseudo-replication</dt><dd>Treating correlated observations (frames) as independent experimental units, inflating n and deflating p-values.</dd>
<dt>Essential dynamics / PCA</dt><dd>Principal component analysis of atomic coordinates; the leading components are the directions of largest positional variance.</dd>
<dt>tICA</dt><dd>Time-lagged independent component analysis: the linear projection maximising autocorrelation at a chosen lag, hence the slowest collective coordinates.</dd>
<dt>Implied timescale</dt><dd>-lag/ln(eigenvalue); an estimate of a relaxation time, trustworthy only when much shorter than the trajectory.</dd>
<dt>Free-energy surface</dt><dd>-kT ln P over a projection; meaningful only for equilibrium sampling, with masked sparse bins and a stated uncertainty.</dd>
<dt>MM/GBSA</dt><dd>End-point method estimating interaction energies frame by frame from a molecular-mechanics energy plus an implicit-solvent term; relative use within one protocol only.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Grossfield A, Zuckerman DM. Quantifying uncertainty and sampling quality in biomolecular simulations. <em>Annu Rep Comput Chem</em>. 2009;5:23-48. <a href="https://doi.org/10.1016/S1574-1400(09)00502-7">doi:10.1016/S1574-1400(09)00502-7</a></li>
<li id="r2">Grossfield A, Patrone PN, Roe DR, Schultz AJ, Siderius DW, Zuckerman DM. Best practices for quantification of uncertainty and sampling quality in molecular simulations. <em>Living J Comput Mol Sci</em>. 2019;1(1):5067. <a href="https://doi.org/10.33011/livecoms.1.1.5067">doi:10.33011/livecoms.1.1.5067</a></li>
<li id="r3">Flyvbjerg H, Petersen HG. Error estimates on averages of correlated data. <em>J Chem Phys</em>. 1989;91(1):461-466. <a href="https://doi.org/10.1063/1.457480">doi:10.1063/1.457480</a></li>
<li id="r4">Chodera JD. A simple method for automated equilibration detection in molecular simulations. <em>J Chem Theory Comput</em>. 2016;12(4):1799-1805. <a href="https://doi.org/10.1021/acs.jctc.5b00784">doi:10.1021/acs.jctc.5b00784</a></li>
<li id="r5">Knapp B, Ospina L, Deane CM. Avoiding false positive conclusions in molecular simulation: the importance of replicas. <em>J Chem Theory Comput</em>. 2018;14(12):6127-6138. <a href="https://doi.org/10.1021/acs.jctc.8b00391">doi:10.1021/acs.jctc.8b00391</a></li>
<li id="r6">Perez-Hernandez G, Paul F, Giorgino T, De Fabritiis G, Noe F. Identification of slow molecular order parameters for Markov model construction. <em>J Chem Phys</em>. 2013;139(1):015102. <a href="https://doi.org/10.1063/1.4811489">doi:10.1063/1.4811489</a></li>
<li id="r7">Michaud-Agrawal N, Denning EJ, Woolf TB, Beckstein O. MDAnalysis: a toolkit for the analysis of molecular dynamics simulations. <em>J Comput Chem</em>. 2011;32(10):2319-2327. <a href="https://doi.org/10.1002/jcc.21787">doi:10.1002/jcc.21787</a></li>
<li id="r8">McGibbon RT, Beauchamp KA, Harrigan MP, et al. MDTraj: a modern open library for the analysis of molecular dynamics trajectories. <em>Biophys J</em>. 2015;109(8):1528-1532. <a href="https://doi.org/10.1016/j.bpj.2015.08.015">doi:10.1016/j.bpj.2015.08.015</a></li>
<li id="r9">Genheden S, Ryde U. The MM/PBSA and MM/GBSA methods to estimate ligand-binding affinities. <em>Expert Opin Drug Discov</em>. 2015;10(5):449-461. <a href="https://doi.org/10.1517/17460441.2015.1032936">doi:10.1517/17460441.2015.1032936</a></li>
<li id="r10">Eastman P, Swails J, Chodera JD, et al. OpenMM 7: rapid development of high performance algorithms for molecular dynamics. <em>PLoS Comput Biol</em>. 2017;13(7):e1005659. <a href="https://doi.org/10.1371/journal.pcbi.1005659">doi:10.1371/journal.pcbi.1005659</a></li>
<li id="r11">Eastman P, Galvelis R, Pelaez RP, et al. OpenMM 8: molecular dynamics simulation with machine learning potentials. <em>J Phys Chem B</em>. 2024;128(1):109-116. <a href="https://doi.org/10.1021/acs.jpcb.3c06662">doi:10.1021/acs.jpcb.3c06662</a></li>
<li id="r12">Maier JA, Martinez C, Kasavajhala K, Wickstrom L, Hauser KE, Simmerling C. ff14SB: improving the accuracy of protein side chain and backbone parameters from ff99SB. <em>J Chem Theory Comput</em>. 2015;11(8):3696-3713. <a href="https://doi.org/10.1021/acs.jctc.5b00255">doi:10.1021/acs.jctc.5b00255</a></li>
<li id="r13">Neidigh JW, Fesinmeyer RM, Andersen NH. Designing a 20-residue protein. <em>Nat Struct Biol</em>. 2002;9(6):425-430. <a href="https://doi.org/10.1038/nsb798">doi:10.1038/nsb798</a></li>
<li id="r14">Qiu L, Pabit SA, Roitberg AE, Hagen SJ. Smaller and faster: the 20-residue Trp-cage protein folds in 4 micros. <em>J Am Chem Soc</em>. 2002;124(44):12952-12953. <a href="https://doi.org/10.1021/ja0273851">doi:10.1021/ja0273851</a></li>
<li id="r15">Kabsch W, Sander C. Dictionary of protein secondary structure: pattern recognition of hydrogen-bonded and geometrical features. <em>Biopolymers</em>. 1983;22(12):2577-2637. <a href="https://doi.org/10.1002/bip.360221211">doi:10.1002/bip.360221211</a></li>
<li id="r16">Amadei A, Linssen ABM, Berendsen HJC. Essential dynamics of proteins. <em>Proteins</em>. 1993;17(4):412-425. <a href="https://doi.org/10.1002/prot.340170408">doi:10.1002/prot.340170408</a></li>
<li id="r17">Balsera MA, Wriggers W, Oono Y, Schulten K. Principal component analysis and long time protein dynamics. <em>J Phys Chem</em>. 1996;100(7):2567-2572. <a href="https://doi.org/10.1021/jp9536920">doi:10.1021/jp9536920</a></li>
<li id="r18">Hess B. Convergence of sampling in protein simulations. <em>Phys Rev E</em>. 2002;65(3):031910. <a href="https://doi.org/10.1103/PhysRevE.65.031910">doi:10.1103/PhysRevE.65.031910</a></li>
<li id="r19">Shaw DE, Maragakis P, Lindorff-Larsen K, et al. Atomic-level characterization of the structural dynamics of proteins. <em>Science</em>. 2010;330(6002):341-346. <a href="https://doi.org/10.1126/science.1187409">doi:10.1126/science.1187409</a></li>
<li id="r20">Lindorff-Larsen K, Piana S, Dror RO, Shaw DE. How fast-folding proteins fold. <em>Science</em>. 2011;334(6055):517-520. <a href="https://doi.org/10.1126/science.1208351">doi:10.1126/science.1208351</a></li>
<li id="r21">Zuckerman DM. Equilibrium sampling in biomolecular simulations. <em>Annu Rev Biophys</em>. 2011;40:41-62. <a href="https://doi.org/10.1146/annurev-biophys-042910-155255">doi:10.1146/annurev-biophys-042910-155255</a></li>
<li id="r22">Prinz JH, Wu H, Sarich M, et al. Markov models of molecular kinetics: generation and validation. <em>J Chem Phys</em>. 2011;134(17):174105. <a href="https://doi.org/10.1063/1.3565032">doi:10.1063/1.3565032</a></li>
<li id="r23">Husic BE, Pande VS. Markov state models: from an art to a science. <em>J Am Chem Soc</em>. 2018;140(7):2386-2396. <a href="https://doi.org/10.1021/jacs.7b12191">doi:10.1021/jacs.7b12191</a></li>
<li id="r24">Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. <em>J R Stat Soc B</em>. 1995;57(1):289-300. <a href="https://doi.org/10.1111/j.2517-6161.1995.tb02031.x">doi:10.1111/j.2517-6161.1995.tb02031.x</a></li>
<li id="r25">Welch BL. The generalization of "Student's" problem when several different population variances are involved. <em>Biometrika</em>. 1947;34(1-2):28-35. <a href="https://doi.org/10.1093/biomet/34.1-2.28">doi:10.1093/biomet/34.1-2.28</a></li>
<li id="r26">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r27">Roe DR, Cheatham TE. PTRAJ and CPPTRAJ: software for processing and analysis of molecular dynamics trajectory data. <em>J Chem Theory Comput</em>. 2013;9(7):3084-3095. <a href="https://doi.org/10.1021/ct400341p">doi:10.1021/ct400341p</a></li>
<li id="r28">Laio A, Parrinello M. Escaping free-energy minima. <em>Proc Natl Acad Sci USA</em>. 2002;99(20):12562-12566. <a href="https://doi.org/10.1073/pnas.202427399">doi:10.1073/pnas.202427399</a></li>
<li id="r29">Braun E, Gilmer J, Mayes HB, et al. Best practices for foundations in molecular simulations. <em>Living J Comput Mol Sci</em>. 2019;1(1):5957. <a href="https://doi.org/10.33011/livecoms.1.1.5957">doi:10.33011/livecoms.1.1.5957</a></li>
<li id="r30">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r31">Shirts MR, Chodera JD. Statistically optimal analysis of samples from multiple equilibrium states. <em>J Chem Phys</em>. 2008;129(12):124105. <a href="https://doi.org/10.1063/1.2978177">doi:10.1063/1.2978177</a></li>
<li id="r32">Bussi G, Donadio D, Parrinello M. Canonical sampling through velocity rescaling. <em>J Chem Phys</em>. 2007;126(1):014101. <a href="https://doi.org/10.1063/1.2408420">doi:10.1063/1.2408420</a></li>
<li id="r33">Efron B. Bootstrap methods: another look at the jackknife. <em>Ann Stat</em>. 1979;7(1):1-26. <a href="https://doi.org/10.1214/aos/1176344552">doi:10.1214/aos/1176344552</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Leach AR. <em>Molecular Modelling: Principles and Applications</em>. 2nd ed. Pearson; 2001. - Chapters 6-7 for the mechanics behind the trajectories analysed here.</li>
<li>Tuckerman ME. <em>Statistical Mechanics: Theory and Molecular Simulation</em>. 2nd ed. Oxford University Press; 2023. - Ensembles, thermostats and the meaning of a free-energy surface.</li>
<li>Wasserman L. <em>All of Statistics</em>. Springer; 2004. - Chapter 8 for the bootstrap used on blocks here.</li>
<li>Shumway RH, Stoffer DS. <em>Time Series Analysis and Its Applications: With R Examples</em>. 4th ed. Springer; 2017. - The autocorrelation machinery of section 2, from the time-series side.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>LiveCoMS best-practices collection for molecular simulation: <a href="https://livecomsjournal.org/">livecomsjournal.org</a>.</li>
<li>pymbar, the reference implementation of equilibration detection and statistical inefficiency: <a href="https://pymbar.readthedocs.io/">pymbar.readthedocs.io</a>.</li>
<li>OpenMM user guide, the engine that produced these trajectories: <a href="https://docs.openmm.org/">docs.openmm.org</a>.</li>
<li>CRAN Task View "Time Series Analysis", for the estimators of section 2: <a href="https://cran.r-project.org/view=TimeSeries">cran.r-project.org/view=TimeSeries</a>.</li>
<li><code>bio3d</code>, an R package for structural bioinformatics and PCA of trajectories (not installed in this course environment): <a href="http://thegrantlab.org/bio3d/">thegrantlab.org/bio3d</a>.</li>
</ul>


